# BUỔI 1 — NOTEBOOK 02: LÀM SẠCH, REGEX, TỔNG HỢP & HỢP NHẤT DỮ LIỆU

> **Mục tiêu Notebook:** Dựa trên các lỗi đã phát hiện ở `01_EDA.ipynb`, tiến hành làm sạch toàn diện 5 tệp dữ liệu bằng `NumPy`, `Pandas` và `Regex`, tổng hợp chỉ số hành vi theo `Student_ID`, ghép bảng, tách riêng tập sinh viên mới chưa có điểm và xuất ra 2 tệp sạch:
> - `../datasets/historical_students.csv` (Dữ liệu lịch sử đầy đủ nhãn)
> - `../datasets/new_students.csv` (Nhóm sinh viên mới chưa có điểm thi để dự báo ở Buổi 3)

---
## Bước 1: Import thư viện & Nạp 5 bảng dữ liệu thô

**Yêu cầu:**
- Import `numpy` và `pandas`.
- Đọc lại 5 tệp CSV từ `../datasets/` kèm `na_values`.

In [ ]:
# TODO: Import thư viện và nạp 5 tệp CSV thô


---
## Bước 2: Làm sạch từng bảng dữ liệu

### 2.1. Làm sạch bảng `students.csv`
**Yêu cầu:**
1. Chuẩn hóa `Student_ID` (`.str.strip().str.upper()`) và loại bỏ trùng lặp theo `Student_ID` (nếu có).
2. Chuẩn hóa `Full_Name`: dùng Regex loại bỏ ký tự đặc biệt thừa, gộp nhiều khoảng trắng (`\s+`) thành 1 dấu cách, `.str.strip().str.title()`.
3. Chuyển `Date_of_Birth` sang kiểu `datetime` bằng `pd.to_datetime(..., format='mixed', errors='coerce')`.
4. Rút trích phần số của `Grade_Level` (ví dụ `'Grade 3'` $\rightarrow$ `3`) bằng `.str.extract(r'(\d+)')` và ép kiểu `int`.
5. Xử lý cột `Emergency_Contact` (điền `'Unknown'` cho giá trị thiếu hoặc xóa cột nếu không dùng trong mô hình hóa).

In [ ]:
# TODO: Làm sạch bảng students


### 2.2. Làm sạch bảng `attendance.csv`
**Yêu cầu:**
1. Loại bỏ các dòng trùng lặp hoàn toàn bằng `.drop_duplicates()` (kiểm tra thêm nếu muốn xử lý trùng theo khóa `['Student_ID', 'Date', 'Subject']`).
2. Chuẩn hóa cột `Date` sang kiểu `datetime`.
3. Dùng `.str.strip().str.lower()` kết hợp **Regex** và `np.select()` để chuẩn hóa toàn bộ biến thể trong `Attendance_Status` về đúng 3 nhãn chuẩn: `'Present'`, `'Late'`, `'Absent'`.
4. Tạo cột cờ nhị phân `is_present` (`1` nếu `'Present'`, `0` nếu ngược lại) và `is_late` (`1` nếu `'Late'`, `0` nếu ngược lại) để chuẩn bị tổng hợp.
5. Kiểm tra lại kết quả bằng `.value_counts()`.

In [ ]:
# TODO: Xóa trùng lặp và chuẩn hóa Attendance_Status về 3 nhóm (Present, Late, Absent)


### 2.3. Làm sạch bảng `homework.csv`
**Yêu cầu:**
1. Biến các chuỗi toàn khoảng trắng `' '` trong `Guardian_Signature` thành `np.nan` bằng `.replace(r'^\s*$', np.nan, regex=True)` rồi xử lý (`.fillna('Unknown')` hoặc quy đổi nhị phân).
2. Chuẩn hóa `Due_Date` sang kiểu `datetime` với `format='mixed'`.
3. Dùng **Regex** (`.str.contains(r'done|✔|✅', regex=True, na=False)`) trên cột `Status` (sau khi `.str.strip().str.lower()`) để tạo biến nhị phân `is_done` (`1` = đã làm, `0` = chưa làm).
4. Quy đổi cột `Grade_Feedback` (`A+`, `A`, `B`, `B-`, `C`, `C-`, `D`, `F`) sang điểm số tương ứng bằng `.map(...)` để tạo cột `hw_gpa`.

In [ ]:
# TODO: Làm sạch bảng homework (Guardian_Signature, Due_Date, Status, Grade_Feedback)


### 2.4. Làm sạch bảng `performance.csv`
**Yêu cầu:**
1. Dùng **Regex** `r'[^0-9.-]'` với `.str.replace(..., regex=True)` để loại bỏ ký tự `%` (hoặc chữ thừa) trong cột `Homework_Completion_%`, sau đó ép kiểu số bằng `pd.to_numeric(..., errors='coerce')`.
2. Xử lý các giá trị ngoại lai (ví dụ điểm `> 100` hoặc tỷ lệ `-5`) bằng `.clip(lower=0, upper=100)` cho cả `Exam_Score` và `Homework_Completion_%`.
3. Kiểm tra lại `.describe()` để chắc chắn mọi giá trị đã nằm gọn trong đoạn `[0, 100]`.
4. Xóa cột `Teacher_Comments` bằng `.drop(columns=...)` nếu không dùng đến.

In [ ]:
# TODO: Làm sạch Homework_Completion_% bằng Regex và xử lý ngoại lai bằng .clip(0, 100)


### 2.5. Làm sạch bảng `teacher_parent_communication.csv`
**Yêu cầu:**
1. Chuẩn hóa `Date` sang kiểu `datetime` và loại bỏ bản ghi trùng lặp (nếu có).
2. Xóa cột văn bản dài `Message_Content` nếu chỉ cần đếm số lần tương tác.

In [ ]:
# TODO: Làm sạch bảng teacher_parent_communication


---
## Bước 3: Tổng hợp chỉ số hành vi theo `Student_ID` (`groupby` + `agg`)

Vì các bảng nhật ký có nhiều dòng cho mỗi sinh viên (quan hệ 1–nhiều), ta phải gom nhóm theo `Student_ID` về đúng **1 dòng / 1 sinh viên** trước khi ghép bảng.

**Yêu cầu tạo 4 bảng tổng hợp (nhớ `.reset_index()`):**
1. `att_agg`: `total_sessions` (số buổi), `attendance_rate` (% có mặt), `late_rate` (% đi muộn).
2. `hw_agg`: `hw_assigned` (số bài được giao), `hw_done_rate` (% hoàn thành BTVN), `avg_hw_gpa` (điểm BTVN quy đổi TB).
3. `comm_agg`: `comm_count` (tổng số lần trao đổi với phụ huynh).
4. `perf_agg`: `final_score` (điểm thi TB trên các môn), `avg_hw` (tỷ lệ hoàn thành BTVN TB từ bảng `performance`).

In [ ]:
# TODO: Dùng .groupby('Student_ID').agg(...).reset_index() tổng hợp 4 bảng nhật ký


---
## Bước 4: Hợp nhất 5 bảng (`pd.merge`) & Xử lý thiếu cho biến đầu vào

**Yêu cầu:**
1. Dùng `pd.merge(..., on='Student_ID', how='left')` xuất phát từ bảng `students` để ghép lần lượt `att_agg`, `hw_agg`, `comm_agg`, `perf_agg`.
2. Kiểm tra số dòng sau khi ghép: đảm bảo `len(merged_df) == len(students)` (không bị nhân bản dòng).
3. Sắp xếp tăng dần theo `Student_ID` (`.sort_values('Student_ID').reset_index(drop=True)`).
4. Xử lý `NaN` cho các **biến đầu vào** phát sinh sau khi `left merge`:
   - `comm_count` bị `NaN` (không có log liên lạc) $\rightarrow$ điền `0`.
   - Các cột từ `homework` bị `NaN` (91 sinh viên không có log BTVN) $\rightarrow$ điền `median` (hoặc `0` tùy lập luận của bạn, ghi rõ lý do).
5. **Tuyệt đối KHÔNG** điền `mean`/`median` vào cột `final_score`!

In [ ]:
# TODO: Merge 5 bảng theo Student_ID, kiểm tra số dòng và xử lý NaN cho các biến đầu vào


---
## Bước 5: Tách tập `new_students` & Tạo biến mục tiêu

**Yêu cầu:**
1. Tách nhóm sinh viên chưa có điểm thi (`merged_df['final_score'].isna()`) thành DataFrame `new_students` (đây là 576 sinh viên mới cần dự báo ở Buổi 3).
2. Giữ nhóm đã có điểm (`merged_df['final_score'].notna()`) thành DataFrame `historical_students`.
3. Trên `historical_students`, tạo biến mục tiêu phân loại nhị phân `is_at_risk`:
   - Ví dụ: `1` nếu `final_score < 65` (nguy cơ trượt môn), `0` nếu `final_score >= 65` (an toàn).
   - Kiểm tra tỷ lệ phân bố 2 lớp bằng `.value_counts(normalize=True)`.

In [ ]:
# TODO: Tách historical_students và new_students; tạo biến mục tiêu is_at_risk


---
## Bước 6: Kiểm tra lần cuối & Xuất dữ liệu sạch ra CSV

**Yêu cầu:**
- Kiểm tra lại `.shape`, `.dtypes`, `.isna().sum()` của `historical_students` và `new_students`.
- Lưu ra `../datasets/historical_students.csv` và `../datasets/new_students.csv` với `index=False`.

In [ ]:
# TODO: Kiểm tra chất lượng lần cuối và xuất 2 file CSV sạch (index=False)
